# 02 — Estrategias de recuperación y RAG agéntico

Este notebook demuestra las cuatro estrategias de recuperación disponibles en el sistema
y muestra cómo la capa agéntica selecciona automáticamente la más adecuada.

| Estrategia | Cuándo usarla | Cómo funciona |
|---|---|---|
| **Vector search** | Consultas semánticas / conceptuales | Similitud coseno sobre embeddings de chunks |
| **Full-text search** | Consultas por palabras clave exactas | Índice Lucene sobre el texto de los chunks |
| **Hybrid search** | Lo mejor de ambos mundos | Combina y reordena resultados vectoriales + full-text |
| **Text2Cypher** | Conteo, agregación, traversal del grafo | El LLM genera una consulta Cypher a partir de lenguaje natural |

**Requisito previo:** Ejecutar `01_ingestion_demo.ipynb` primero para cargar los datos de Einstein en Neo4j.

## 1. Configuración

In [1]:
import sys
sys.path.append('..')

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.retrieval.vector_retriever import VectorRetriever, HybridRetriever
from graphrag.retrieval.fulltext_retriever import FullTextRetriever
from graphrag.retrieval.text2cypher import Text2CypherRetriever
from graphrag.agents import AgenticRAG

neo4j = Neo4jManager()
print("Conectado a Neo4j.")

Conectado a Neo4j.


## 2. Vector search — similitud semántica

La consulta se embede con el mismo modelo usado en la ingestión (`nomic-embed-text`).
Neo4j devuelve los `top_k` chunks cuyos vectores de embedding son más cercanos al vector de la consulta.

Ideal para: *"What did Einstein work on?"*, *"Tell me about relativity"*

In [2]:
vector_retriever = VectorRetriever(neo4j)

results = vector_retriever.retrieve("What did Einstein work on?")

print(f"Vector search — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f}")
    print(f"   {r['text'][:180]}...\n")

Vector search — 4 resultados

1. score=0.770
   Albert Einstein was a German-born theoretical physicist who is widely held
to be one of the greatest and most influential scientists of all time.
He developed the theory of relativ...

2. score=0.757
   several groundbreaking papers, including
his work on special relativity in 1905. He later moved to Princeton,
New Jersey, where he worked at the Institute for Advanced Study.

The ...

3. score=0.754
   and made important contributions to
quantum mechanics. Einstein was born in Ulm, Germany in 1879.

Einstein worked at the Swiss Patent Office in Bern from 1902 to 1909.
During this...

4. score=0.739
   published in 1915, revolutionized our
understanding of gravity and space-time. Einstein received the Nobel Prize
in Physics in 1921 for his explanation of the photoelectric effect....



## 3. Full-text search — coincidencia por palabras clave

Utiliza un índice Lucene de texto completo sobre el texto de los chunks.
Rápido y preciso para términos específicos, pero no detecta paráfrasis.

Ideal para: *"Nobel Prize"*, *"photoelectric effect"*

In [3]:
fulltext_retriever = FullTextRetriever(neo4j)

results = fulltext_retriever.retrieve("Nobel Prize photoelectric")

print(f"Full-text search — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f}")
    print(f"   {r['text'][:180]}...\n")

Full-text search — 1 resultados

1. score=1.769
   published in 1915, revolutionized our
understanding of gravity and space-time. Einstein received the Nobel Prize
in Physics in 1921 for his explanation of the photoelectric effect....



## 4. Hybrid search — combinado y reordenado

Ejecuta búsqueda vectorial y de texto completo en paralelo, normaliza las puntuaciones dentro de cada rama,
luego hace la unión y reordena por la puntuación normalizada máxima.

Generalmente la estrategia más robusta para consultas abiertas.

In [4]:
hybrid_retriever = HybridRetriever(neo4j)

results = hybrid_retriever.retrieve("Nobel Prize photoelectric")

print(f"Hybrid search — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f}")
    print(f"   {r['text'][:180]}...\n")

Hybrid search — 4 resultados

1. score=1.000
   published in 1915, revolutionized our
understanding of gravity and space-time. Einstein received the Nobel Prize
in Physics in 1921 for his explanation of the photoelectric effect....

2. score=0.955
   Albert Einstein was a German-born theoretical physicist who is widely held
to be one of the greatest and most influential scientists of all time.
He developed the theory of relativ...

3. score=0.909
   and made important contributions to
quantum mechanics. Einstein was born in Ulm, Germany in 1879.

Einstein worked at the Swiss Patent Office in Bern from 1902 to 1909.
During this...

4. score=0.889
   several groundbreaking papers, including
his work on special relativity in 1905. He later moved to Princeton,
New Jersey, where he worked at the Institute for Advanced Study.

The ...



## 5. Text2Cypher — lenguaje natural a consulta de grafo

Un LLM traduce la pregunta en lenguaje natural a una consulta Cypher,
que se ejecuta directamente contra Neo4j.

Imprescindible para preguntas que requieren **conteo**, **agregación** o
**traversal multi-salto** del grafo — cosas que la búsqueda vectorial no puede responder.

Los ejemplos *few-shot* ayudan al LLM a aprender el esquema del grafo.

In [5]:
text2cypher = Text2CypherRetriever(neo4j)

# Few-shot examples usando el esquema de dominio real
text2cypher.add_few_shot_example(
    "What institutions did Einstein work at?",
    "MATCH (p:Person {name: 'Albert Einstein'})-[r:WORKED_AT]->(i:Institution) "
    "RETURN i.name, r.from_year, r.to_year, r.role"
)
text2cypher.add_few_shot_example(
    "Which theories did Einstein develop?",
    "MATCH (p:Person {name: 'Albert Einstein'})-[r:DEVELOPED]->(t:ScientificTheory) "
    "RETURN t.name, r.year ORDER BY r.year"
)
text2cypher.add_few_shot_example(
    "How many Person nodes are in the graph?",
    "MATCH (p:Person) RETURN count(p) AS count"
)


In [6]:
# Consulta de conteo sobre nodo de dominio real
cypher, results = text2cypher.retrieve("How many Person nodes are in the graph?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.schema.nodeTypeProperties()\n        YIELD nodeType, propertyName, propertyTypes\n        WITH nodeType, collect({property: propertyName, type: propertyTypes[0]}) as properties\n        RETURN {labels: nodeType, properties: properties} AS output\

Error ejecutando Cypher: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'Okay': expected 'ALTER', 'ORDER BY', 'CALL', 'USING PERIODIC COMMIT', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FINISH', 'FOREACH', 'GRANT', 'INSERT', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE' or 'WITH' (line 1, column 1 (offset: 0))
"Okay, let's see. The user is asking how many Person nodes are in the graph. I need to convert that into a Cypher query."
 ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}
Query generada: Okay, let's see. The user is asking how many Person nodes are in the graph. I need to convert that into a Cypher query.

First, looking at the graph schema provided. The node labels include :Perso

In [7]:
# Traversal: qué instituciones aparecen en el grafo
cypher, results = text2cypher.retrieve(
    "Which institutions are connected to Einstein and in which years?"
)

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.schema.nodeTypeProperties()\n        YIELD nodeType, propertyName, propertyTypes\n        WITH nodeType, collect({property: propertyName, type: propertyTypes[0]}) as properties\n        RETURN {labels: nodeType, properties: properties} AS output\

Error ejecutando Cypher: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'Okay': expected 'ALTER', 'ORDER BY', 'CALL', 'USING PERIODIC COMMIT', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FINISH', 'FOREACH', 'GRANT', 'INSERT', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE' or 'WITH' (line 1, column 1 (offset: 0))
"Okay, let's tackle this problem. The user is asking: "Which institutions are connected to Einstein and in which years?""
 ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}
Query generada: Okay, let's tackle this problem. The user is asking: "Which institutions are connected to Einstein and in which years?" 

First, I need to figure out how Einstein is represented in the graph. From

## 6. AgenticRAG — enrutamiento automático

La capa agéntica envuelve los cuatro recuperadores tras una única llamada `answer()`.
Un router LLM lee la pregunta, elige la herramienta más adecuada, recupera contexto,
genera una respuesta y luego la critica, iterando si es necesario.

```
pregunta ──▶ Router ──▶ Herramienta ──▶ Recuperación ──▶ Respuesta ──▶ Crítica
                 ▲                                                          │
                 └───────────────── reintento si es necesario ──────────────┘
```

In [8]:
rag = AgenticRAG(neo4j)

In [9]:
# Pregunta factual simple — el router elegirá vector o hybrid search
result = rag.answer("Where was Einstein born?")

print("Question:", result['question'])
print("Answer:", result['answer'])
tool = result['iterations'][-1]['retrieval']['tool']
print(f"Herramienta usada: {tool}")

Question: Where was Einstein born?
Answer: Let me analyze the context to find where Einstein was born.

Looking at the context:

[1] Albert Einstein was a German-born theoretical physicist who is widely held
to be one of the greatest and most influential scientists of all time.
He developed the theory of relativity and made important

[2] and made important contributions to
quantum mechanics. Einstein was born in Ulm, Germany in 1879.

[3] several groundbreaking papers, including
his work on special relativity in 1905. He later moved to Princeton,
New Jersey, where he worked at the Institute for Advanced Study.

The theory of general relativity, published in 1915,

[4] published in 1915, revolutionized our
understanding of gravity and space-time. Einstein received the Nobel Prize
in Physics in 1921 for his explanation of the photoelectric effect.

I can see that in context [2], it states: "Einstein was born in Ulm, Germany in 1879."

This directly answers the question about where Einst

In [10]:
# Pregunta de agregación — el router debería elegir text2cypher
result = rag.answer("List all locations mentioned in the documents")

print("Question:", result['question'])
print("Answer:", result['answer'])

last_iter = result['iterations'][-1]
tool = last_iter['retrieval']['tool']
reasoning = last_iter['retrieval'].get('routing_decision', {}).get('reasoning', '')
print(f"\nHerramienta usada: {tool}")
print(f"Razonamiento del router: {reasoning}")

Question: List all locations mentioned in the documents
Answer: Let me carefully analyze the context to find all locations mentioned.

First, I'll go through each source:

[1] mentions:
- Princeton, New Jersey (where he moved to work at the Institute for Advanced Study)

[2] mentions:
- Ulm, Germany (where Einstein was born in 1879)
- Bern (where he worked at the Swiss Patent Office)

[3] doesn't mention any specific locations

[4] mentions:
- Germany (where Einstein was born)

Let me list all the locations mentioned:

1. Princeton, New Jersey
2. Ulm, Germany
3. Bern (city in Switzerland)
4. Germany (country)

Wait, I need to be careful about what's considered a "location" here. The question asks for "locations mentioned in the documents". Let me check if I'm missing anything.

From [1]: "He later moved to Princeton, New Jersey, where he worked at the Institute for Advanced Study." → Princeton, New Jersey

From [2]: "Einstein was born in Ulm, Germany in 1879." → Ulm, Germany
"worked at

AttributeError: 'RouterDecision' object has no attribute 'get'

## 7. Conversación multi-turno

El agente mantiene el historial de conversación para que las preguntas de seguimiento
resuelvan pronombres correctamente.

In [11]:
rag.reset_conversation()

r1 = rag.answer("What theory did Einstein develop?")
print(f"Q1: {r1['question']}")
print(f"A1: {r1['answer']}\n")

# 'it' hace referencia a la teoría de la relatividad — el agente usa el historial
r2 = rag.answer("When was it published?")
print(f"Q2: {r2['question']}")
print(f"A2: {r2['answer']}")

Q1: What theory did Einstein develop?
A1: Let me analyze the context to answer the question about what theory Einstein developed.

The context provides information about Einstein:

[1] Albert Einstein was a German-born theoretical physicist who is widely held to be one of the greatest and most influential scientists of all time. He developed the theory of relativity and made important...

[2] several groundbreaking papers, including his work on special relativity in 1905. He later moved to Princeton, New Jersey, where he worked at the Institute for Advanced Study.

The theory of general relativity, published in 1915,

[3] and made important contributions to quantum mechanics. Einstein was born in Ulm, Germany in 1879.

Einstein worked at the Swiss Patent Office in Bern from 1902 to 1909.
During this time, he published several groundbreaking

[4] published in 1915, revolutionized our understanding of gravity and space-time. Einstein received the Nobel Prize in Physics in 1921 for his ex

In [ ]:
neo4j.close()